In [3]:
import os
import json
import glob
import re

# 用于自然排序的辅助函数 (确保 fold_1, fold_2, ... fold_10 顺序正确)
def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower()
            for text in re.split('([0-9]+)', s)]

print("库导入完成。")

库导入完成。


In [4]:
def merge_folds_from_folder(folder_path, output_name="All_Folds_Summary"):
    # 1. 获取所有 JSON 文件
    search_path = os.path.join(folder_path, "*.json")
    files = glob.glob(search_path)
    
    # 按数字顺序排序 (避免 1, 10, 11, 2... 的情况)
    files.sort(key=lambda x: natural_sort_key(os.path.basename(x)))
    
    if not files:
        print(f"❌ 错误：在路径 '{folder_path}' 下没有找到 .json 文件。")
        return

    print(f"📂 找到 {len(files)} 个文件，准备合并...")
    
    # 存储合并后的数据
    merged_data_list = []
    
    # 2. 生成适合 AI 阅读的 TXT 文本
    txt_output_path = os.path.join(folder_path, f"{output_name}.txt")
    json_output_path = os.path.join(folder_path, f"{output_name}.json")
    
    with open(txt_output_path, 'w', encoding='utf-8') as txt_file:
        # 写入文件头，告诉 AI 这是什么
        txt_file.write(f"Dataset Summary: {len(files)} Folds Cross-Validation Results\n")
        txt_file.write("Task: Analysis of training stability across folds.\n")
        txt_file.write("="*50 + "\n\n")

        for file_path in files:
            file_name = os.path.basename(file_path)
            
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = json.load(f)
                    
                    # 添加到 JSON 列表
                    merged_data_list.append({
                        "fold_name": file_name,
                        "data": content
                    })
                    
                    # 写入 TXT (AI 阅读版)
                    txt_file.write(f"=== SOURCE: {file_name} ===\n")
                    # 将 JSON 转换为紧凑的字符串写入
                    txt_file.write(json.dumps(content, ensure_ascii=False)) 
                    txt_file.write("\n\n" + "-"*30 + "\n\n")
                    
            except Exception as e:
                print(f"⚠️ 读取文件 {file_name} 时出错: {e}")

    # 3. 保存结构化 JSON (可选，供程序使用)
    with open(json_output_path, 'w', encoding='utf-8') as json_file:
        json.dump(merged_data_list, json_file, indent=2)

    print(f"✅ 合并完成！")
    print(f"📄 AI 分析专用文件: {txt_output_path}")
    print(f"📊 结构化数据文件: {json_output_path}")

In [5]:
# ================= 配置区域 =================
# 在这里输入你的文件夹路径 (Windows路径请使用反斜杠 \ 或在字符串前加 r)
folder_path = "/Users/jannik/Library/CloudStorage/OneDrive-个人/Study-Materials-and-Documents/Master/Masterarbeit/撰写/data/softlabel等数据/baseline history" 
# ===========================================

# 运行合并
merge_folds_from_folder(folder_path)

📂 找到 39 个文件，准备合并...
✅ 合并完成！
📄 AI 分析专用文件: /Users/jannik/Library/CloudStorage/OneDrive-个人/Study-Materials-and-Documents/Master/Masterarbeit/撰写/data/softlabel等数据/baseline history/All_Folds_Summary.txt
📊 结构化数据文件: /Users/jannik/Library/CloudStorage/OneDrive-个人/Study-Materials-and-Documents/Master/Masterarbeit/撰写/data/softlabel等数据/baseline history/All_Folds_Summary.json


In [8]:
import os
import glob
import json
import re

# ================= 配置区域 =================
# 你的文件夹路径
folder_path = "/Users/jannik/Library/CloudStorage/OneDrive-个人/Study-Materials-and-Documents/Master/Masterarbeit/撰写/data/softlabel等数据/baseline per class 实验数据" 

# 要匹配的文件名模式
file_pattern = "per_class_metrics_test*.json"

# 输出文件名
output_json_name = "All_Per_Class_Metrics_Summary.json"
output_txt_name = "All_Per_Class_Metrics_Summary.txt"
# ===========================================

def natural_sort_key(s):
    """自然排序：确保 test1, test2 ... test10 顺序正确"""
    return [int(text) if text.isdigit() else text.lower()
            for text in re.split('([0-9]+)', s)]

def merge_to_json_and_txt():
    # 1. 查找文件
    search_path = os.path.join(folder_path, file_pattern)
    files = glob.glob(search_path)
    
    # 2. 排序
    files.sort(key=lambda x: natural_sort_key(os.path.basename(x)))
    
    if not files:
        print(f"❌ 未找到匹配的文件: {file_pattern}")
        print(f"请检查路径是否正确: {search_path}")
        return

    print(f"📂 找到 {len(files)} 个文件，开始合并...")
    
    all_folds_data = []
    
    # === 关键修改：拼接完整的输出路径 ===
    full_txt_path = os.path.join(folder_path, output_txt_name)
    full_json_path = os.path.join(folder_path, output_json_name)
    
    # 打开 TXT 文件准备写入
    with open(full_txt_path, 'w', encoding='utf-8') as txt_outfile:
        txt_outfile.write(f"DATASET SUMMARY: {len(files)} Folds Metrics\n{'='*50}\n\n")
        
        for file_path in files:
            filename = os.path.basename(file_path)
            try:
                with open(file_path, 'r', encoding='utf-8') as infile:
                    data = json.load(infile)
                    
                    # === A. 处理 JSON 合并 ===
                    all_folds_data.append({
                        "fold_name": filename,
                        "metrics": data
                    })
                    
                    # === B. 处理 TXT 合并 ===
                    txt_outfile.write(f"=== SOURCE: {filename} ===\n")
                    txt_outfile.write(json.dumps(data, ensure_ascii=False, indent=2))
                    txt_outfile.write(f"\n\n{'='*30}\n\n")
                    
                    print(f"✅ 已处理: {filename}")
                    
            except Exception as e:
                print(f"⚠️ 读取出错 {filename}: {e}")

    # === C. 保存最终的 JSON 合并文件 ===
    # 使用拼接好的完整路径
    with open(full_json_path, 'w', encoding='utf-8') as json_outfile:
        json.dump(all_folds_data, json_outfile, ensure_ascii=False, indent=2)

    print(f"\n🎉 合并完成！文件已保存到数据文件夹中：")
    print(f"1. 📄 {full_txt_path}")
    print(f"2. 📊 {full_json_path}")

# 运行函数
if __name__ == "__main__":
    merge_to_json_and_txt()

📂 找到 38 个文件，开始合并...
✅ 已处理: per_class_metrics_test1.json
✅ 已处理: per_class_metrics_test2.json
✅ 已处理: per_class_metrics_test3.json
✅ 已处理: per_class_metrics_test4.json
✅ 已处理: per_class_metrics_test5.json
✅ 已处理: per_class_metrics_test6.json
✅ 已处理: per_class_metrics_test7.json
✅ 已处理: per_class_metrics_test8.json
✅ 已处理: per_class_metrics_test9.json
✅ 已处理: per_class_metrics_test10.json
✅ 已处理: per_class_metrics_test11.json
✅ 已处理: per_class_metrics_test12.json
✅ 已处理: per_class_metrics_test13.json
✅ 已处理: per_class_metrics_test14.json
✅ 已处理: per_class_metrics_test15.json
✅ 已处理: per_class_metrics_test16.json
✅ 已处理: per_class_metrics_test17.json
✅ 已处理: per_class_metrics_test18.json
✅ 已处理: per_class_metrics_test19.json
✅ 已处理: per_class_metrics_test20.json
✅ 已处理: per_class_metrics_test21.json
✅ 已处理: per_class_metrics_test22.json
✅ 已处理: per_class_metrics_test23.json
✅ 已处理: per_class_metrics_test24.json
✅ 已处理: per_class_metrics_test25.json
✅ 已处理: per_class_metrics_test26.json
✅ 已处理: per_class_metrics_te